In [24]:
#export MODEL="Your model here
export MODEL="HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_NL"

In [25]:
llama-fit-params -hf $MODEL -ncmoe 99 # This is just for downloading and some initial values to probe

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping

llama_params_fit_impl: projected to use 7411 MiB of device memory vs. 5565 MiB of free device memory
llama_params_fit_impl: cannot meet free memory target of 1024 MiB, need to reduce device memory by 2870 MiB
llama_params_fit_impl: context size reduced from 262144 to 126720 -> need 2873 MiB less memory in total
llama_params_fit_impl: entire model can be fit by reducing context
llama_params_fit: successfully fit params to free device memory
llama_params_fit: fitting params to free memory took 1.99 seconds
main: printing fitted CLI arguments to stdout...
-c 126720 -ngl -1 -ot "blk\.0\.ffn_(up|down|gate|gate_up)_(ch|)exps=CPU,b

In [26]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="512 1024 1440 2048"
FITT="512"
CMOE="99"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            for cmoe in $CMOE; do
                # need to add fitt for nvidia
                OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft -ncmoe $cmoe 2>&1)
                
                #echo "Raw output: $OUTPUT"  # Debug line  
                # Extract context and GPU layers more robustly  
                CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
                NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
                
                if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                    echo "ub: $ub, b: $b, fitt: $ft, ncmoe: $cmoe, ctx: $CTX, ngl: $NGL"  
                else  
                    echo "No valid parameters found"  
                fi
            done
        done
    done  
done 

Testing different batch/ubatch/fitt combinations for HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_NL...
ub: 512, b: 2048, fitt: 512, ncmoe: 99, ctx: 150784, ngl: -1
ub: 1024, b: 2048, fitt: 512, ncmoe: 99, ctx: 121344, ngl: -1
ub: 1440, b: 2048, fitt: 512, ncmoe: 99, ctx: 99584, ngl: -1
ub: 2048, b: 2048, fitt: 512, ncmoe: 99, ctx: 70656, ngl: -1


In [27]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -fitt 512 -ub 512,1024 --mmap 0 -ngl 99 -ncmoe 99 -p 1000,2000 -n 256,512
# these numbers are from headless ubuntu with no GUI

Staring llama-bench for HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_NL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |       fitt |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | ---------: | --------------: | -------------------: |
| qwen35moe 35B.A3B IQ4_NL - 4.5 bpw |  18.41 GiB |    34.66 B | CUDA       |  99 |         99 |      512 |    0 |        512 |          pp1000 |       402.87 ± 14.34 |
| qwen35moe 35B.A3B IQ4_NL - 4.5 bpw |  18.41 GiB |    34.66 B | CUDA       |  99 |         99 |      512 |    0 |        512 |          p

In [ ]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -fitt 512 -ub 1440 --mmap 0 -ngl 99 -ncmoe 99 -p 1000,2000 -n 256,512

Staring llama-bench for HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ4_NL...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |       fitt |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | ---------: | --------------: | -------------------: |


In [17]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -fitt 512 -ub 512,1024 --mmap 0 -ngl 99 -ncmoe 32,42,52,62 -p 1000,2000 -n 256,512

Staring llama-bench for HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive:IQ3_M...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |       fitt |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | ---------: | --------------: | -------------------: |
| qwen35moe 35B.A3B IQ3_S mix - 3.66 bpw |  14.37 GiB |    34.66 B | CUDA       |  99 |         32 |      512 |    0 |        512 |          pp1000 |        483.09 ± 3.65 |
| qwen35moe 35B.A3B IQ3_S mix - 3.66 bpw |  14.37 GiB |    34.66 B | CUDA       |  99 |         32 |      512 |    0 |        512 |    

In [ ]:
llama-fit-params -hf $MODEL -ncmoe 32

In [ ]:
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off --single-turn --prompt "Who are you?"

In [ ]:
echo "Staring llama-server for ${MODEL}..." 
llama-server -hf $MODEL --threads-http 1 -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off -np 1